**1. INSTALLATION & IMPORTS**

In [3]:
!pip install -q transformers accelerate

import os
import random
import csv
import json
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset, random_split

import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode

from transformers import AutoModelForSemanticSegmentation
from transformers import SegformerConfig, SegformerForSemanticSegmentation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/INF8225/Datasets/isic
!ls /content/drive/MyDrive/INF8225/Datasets/pets
!ls /content/drive/MyDrive/INF8225/Datasets/crack


!cp -r /content/drive/MyDrive/INF8225/Datasets/isic /content/
!cp -r /content/drive/MyDrive/INF8225/Datasets/pets /content/
!cp -r /content/drive/MyDrive/INF8225/Datasets/crack /content/


Device: cuda
Mounted at /content/drive
test  train
test  train
test  train


**SEED GLOBAL — REPRODUCTIBILITÉ**

In [4]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

**CONFIGURATION**

In [5]:
########################################
# CONFIGURATION
########################################

DATASET = "isic"  # "isic" | "pets" | "crack"

AUGMENTATION_CONFIG = "baseline"
# "baseline" | "flip" | "flip_rotation" | "full_aug" | "geo_strong" | "extreme_aug"

TRAIN_SIZE = 100
# 50 | 100 | 250 | 500 | "full"

PATIENCE = 10
EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-4

MODEL_NAME = "segformer-b0-from-scratch"


OUTPUT_DIR = "/content/drive/MyDrive/segformer_untrained_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

**DATASETS**

In [6]:
class ISICDataset(Dataset):
    def __init__(self, images_dir, masks_dir, augmentation_config="baseline"):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.augmentation_config = augmentation_config

        self.images = sorted(os.listdir(images_dir))
        self.masks = sorted(os.listdir(masks_dir))

        assert len(self.images) == len(self.masks), "Images et masques doivent correspondre !"

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image, mask = apply_transforms(image, mask, self.augmentation_config, dataset="isic")

        return image, mask


class OxfordPetDataset(Dataset):
    def __init__(self, images_dir, masks_dir, augmentation_config="baseline"):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.augmentation_config = augmentation_config

        self.images = []
        self.masks = []

        for img in sorted(os.listdir(images_dir)):
            mask_name = img.replace(".jpg", ".png")
            mask_path = os.path.join(masks_dir, mask_name)

            if os.path.exists(mask_path):
                self.images.append(img)
                self.masks.append(mask_name)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # Oxford-IIIT Pet : classe 1 = animal, autres = fond
        mask_np = np.array(mask)
        mask_np = (mask_np == 1).astype(np.uint8) * 255
        mask = Image.fromarray(mask_np)

        image, mask = apply_transforms(image, mask, self.augmentation_config, dataset="pets")

        return image, mask


class CrackDataset(Dataset):
    def __init__(self, images_dir, masks_dir, augmentation_config="baseline"):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.augmentation_config = augmentation_config

        self.images = sorted(os.listdir(images_dir))
        self.masks = sorted(os.listdir(masks_dir))

        assert len(self.images) == len(self.masks), "Images et masques doivent correspondre !"

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image, mask = apply_transforms(image, mask, self.augmentation_config, dataset="crack")

        return image, mask

**TRANSFORMS & AUGMENTATIONS**

In [7]:
def apply_transforms(image, mask, augmentation_config="baseline", dataset=None):
    image = transforms.Resize((256, 256))(image)
    mask = transforms.Resize((256, 256), interpolation=InterpolationMode.NEAREST)(mask)

    aug = augmentation_config

    ########################################
    # AUGMENTATIONS SYNCHRONISÉES IMAGE / MASK
    ########################################

    if aug in ["flip", "flip_rotation", "full_aug", "geo_strong", "extreme_aug"]:
        if random.random() < 0.5:
            image = transforms.functional.hflip(image)
            mask = transforms.functional.hflip(mask)

    if aug in ["full_aug", "geo_strong", "extreme_aug"]:
        if random.random() < 0.5:
            image = transforms.functional.vflip(image)
            mask = transforms.functional.vflip(mask)

    if aug in ["flip_rotation", "full_aug", "geo_strong", "extreme_aug"]:
        angle = random.uniform(-25, 25)
        image = transforms.functional.rotate(image, angle)
        mask = transforms.functional.rotate(
            mask,
            angle,
            interpolation=InterpolationMode.NEAREST
        )

    if aug in ["geo_strong", "extreme_aug"]:
        tx = random.uniform(-0.1, 0.1) * 256
        ty = random.uniform(-0.1, 0.1) * 256
        scale = random.uniform(0.8, 1.2)
        shear = random.uniform(-10, 10)

        image = transforms.functional.affine(
            image,
            angle=0,
            translate=(int(tx), int(ty)),
            scale=scale,
            shear=shear
        )

        mask = transforms.functional.affine(
            mask,
            angle=0,
            translate=(int(tx), int(ty)),
            scale=scale,
            shear=shear,
            interpolation=InterpolationMode.NEAREST
        )

    ########################################
    # IMAGE ONLY AUGMENTATIONS
    ########################################

    if aug in ["full_aug", "extreme_aug"]:
        image = transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2
        )(image)

    if aug == "extreme_aug":
        image = transforms.GaussianBlur(kernel_size=3)(image)

    ########################################
    # TENSORS
    ########################################

    image = transforms.ToTensor()(image)

    # Pour rester cohérent avec le code U-Net du coéquipier
    image = transforms.Normalize(
        [0.5, 0.5, 0.5],
        [0.5, 0.5, 0.5]
    )(image)

    mask = transforms.ToTensor()(mask)
    mask = (mask > 0.5).long().squeeze(0)  # [H, W], valeurs 0 ou 1

    return image, mask


def denormalize(image):
    image = image.clone()
    image = image * 0.5 + 0.5
    return image.clamp(0, 1)

**DATASET SELECTION**

In [8]:
DATASET_CLASSES = {
    "isic": ISICDataset,
    "pets": OxfordPetDataset,
    "crack": CrackDataset,
}

DATASET_PATHS = {
    "isic": {
        "train_images": "/content/isic/train/images",
        "train_masks": "/content/isic/train/masks",
        "test_images": "/content/isic/test/images",
        "test_masks": "/content/isic/test/masks",
    },
    "pets": {
        "train_images": "/content/pets/train/images",
        "train_masks": "/content/pets/train/masks",
        "test_images": "/content/pets/test/images",
        "test_masks": "/content/pets/test/masks",
    },
    "crack": {
        "train_images": "/content/crack/train/images",
        "train_masks": "/content/crack/train/masks",
        "test_images": "/content/crack/test/images",
        "test_masks": "/content/crack/test/masks",
    },
}

**BUILD DATALOADERS**

In [9]:
def build_dataloaders(dataset_name, train_size, augmentation_config, batch_size=8):
    if dataset_name not in DATASET_CLASSES:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    DatasetClass = DATASET_CLASSES[dataset_name]
    paths = DATASET_PATHS[dataset_name]

    train_full = DatasetClass(
        paths["train_images"],
        paths["train_masks"],
        augmentation_config=augmentation_config
    )

    test_dataset = DatasetClass(
        paths["test_images"],
        paths["test_masks"],
        augmentation_config="baseline"
    )

    total_train = len(train_full)

    if train_size == "full":
        subset_size = total_train
    else:
        subset_size = min(int(train_size), total_train)

    indices = list(range(subset_size))
    train_subset = Subset(train_full, indices)

    train_len = int(0.85 * len(train_subset))
    val_len = len(train_subset) - train_len

    generator = torch.Generator().manual_seed(SEED)

    train_dataset, val_dataset = random_split(
        train_subset,
        [train_len, val_len],
        generator=generator
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Dataset: {dataset_name}")
    print(f"Augmentation: {augmentation_config}")
    print(f"Train size requested: {train_size}")
    print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

    return train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset

**METRICS — SEGFORMER**

In [10]:
def preds_from_logits(logits, target_size):
    logits = F.interpolate(
        logits,
        size=target_size,
        mode="bilinear",
        align_corners=False
    )
    preds = torch.argmax(logits, dim=1)
    return preds


def iou_score_segformer(preds, masks, smooth=1e-6):
    preds = preds.float()
    masks = masks.float()

    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection

    return (intersection + smooth) / (union + smooth)


def dice_score_segformer(preds, masks, smooth=1e-6):
    preds = preds.float()
    masks = masks.float()

    intersection = (preds * masks).sum()

    return (2 * intersection + smooth) / (preds.sum() + masks.sum() + smooth)


def pixel_accuracy_segformer(preds, masks):
    return (preds == masks).float().mean()


def precision_score_segformer(preds, masks, smooth=1e-6):
    preds = preds.float()
    masks = masks.float()

    tp = (preds * masks).sum()
    fp = (preds * (1 - masks)).sum()

    return (tp + smooth) / (tp + fp + smooth)


def recall_score_segformer(preds, masks, smooth=1e-6):
    preds = preds.float()
    masks = masks.float()

    tp = (preds * masks).sum()
    fn = ((1 - preds) * masks).sum()

    return (tp + smooth) / (tp + fn + smooth)

**MODEL — SEGFORMER**

In [11]:
def build_segformer_model():
    config = SegformerConfig(
        num_labels=2,
        id2label={0: "background", 1: "foreground"},
        label2id={"background": 0, "foreground": 1},

        num_channels=3,
        depths=[2, 2, 2, 2],
        sr_ratios=[8, 4, 2, 1],
        hidden_sizes=[32, 64, 160, 256],
        patch_sizes=[7, 3, 3, 3],
        strides=[4, 2, 2, 2],
        num_attention_heads=[1, 2, 5, 8],
        mlp_ratios=[4, 4, 4, 4],
        hidden_dropout_prob=0.0,
        attention_probs_dropout_prob=0.0,
        classifier_dropout_prob=0.1,
    )

    model = SegformerForSemanticSegmentation(config)
    model.to(device)
    return model


def quick_model_check(model, train_loader):
    images, masks = next(iter(train_loader))
    images = images.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(pixel_values=images, labels=masks)
        logits = outputs.logits
        preds = preds_from_logits(logits, masks.shape[-2:])

    print("Images shape:", images.shape)
    print("Masks shape :", masks.shape)
    print("Logits shape:", logits.shape)
    print("Preds shape :", preds.shape)
    print("Loss test  :", outputs.loss.item())

    assert masks.ndim == 3, "Les masks doivent être [B,H,W]"
    assert preds.shape == masks.shape, "Les prédictions doivent avoir la même shape que les masks"

    print("Vérification SegFormer réussie.")

**TRAIN ONE EXPERIMENT**

In [12]:
def train_one_experiment(dataset_name, train_size, augmentation_config):
    set_seed(SEED)

    exp_name = f"segformer_{dataset_name}_N{train_size}_{augmentation_config}"
    exp_dir = os.path.join(OUTPUT_DIR, exp_name)
    os.makedirs(exp_dir, exist_ok=True)

    train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = build_dataloaders(
        dataset_name=dataset_name,
        train_size=train_size,
        augmentation_config=augmentation_config,
        batch_size=BATCH_SIZE
    )

    model = build_segformer_model()
    quick_model_check(model, train_loader)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    best_iou = 0.0
    patience_counter = 0

    train_losses = []
    val_losses = []
    val_ious = []
    val_dices = []
    val_accs = []

    for epoch in range(EPOCHS):
        ########################################
        # TRAIN
        ########################################

        model.train()
        train_loss = 0.0

        for images, masks in tqdm(train_loader, desc=f"[{exp_name}] Epoch {epoch+1} train"):
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()

            if use_amp:
                with torch.amp.autocast("cuda"):
                    outputs = model(pixel_values=images, labels=masks)
                    loss = outputs.loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            else:
                outputs = model(pixel_values=images, labels=masks)
                loss = outputs.loss
                loss.backward()
                optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        ########################################
        # VALIDATION
        ########################################

        model.eval()

        val_loss = 0.0
        val_iou = 0.0
        val_dice = 0.0
        val_acc = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)

                if use_amp:
                    with torch.amp.autocast("cuda"):
                        outputs = model(pixel_values=images, labels=masks)
                else:
                    outputs = model(pixel_values=images, labels=masks)

                loss = outputs.loss
                logits = outputs.logits

                preds = preds_from_logits(logits, masks.shape[-2:])

                val_loss += loss.item()
                val_iou += iou_score_segformer(preds, masks).item()
                val_dice += dice_score_segformer(preds, masks).item()
                val_acc += pixel_accuracy_segformer(preds, masks).item()

        val_loss /= len(val_loader)
        val_iou /= len(val_loader)
        val_dice /= len(val_loader)
        val_acc /= len(val_loader)

        scheduler.step(val_iou)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_ious.append(val_iou)
        val_dices.append(val_dice)
        val_accs.append(val_acc)

        if val_iou > best_iou:
            best_iou = val_iou
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(exp_dir, "best_segformer.pth"))
        else:
            patience_counter += 1

        print(f"\n[{exp_name}] Epoch {epoch+1}/{EPOCHS}")
        print(f"  Train loss : {train_loss:.4f}")
        print(f"  Val loss   : {val_loss:.4f}")
        print(f"  Val IoU    : {val_iou:.4f}   best={best_iou:.4f}")
        print(f"  Val Dice   : {val_dice:.4f}")
        print(f"  Val Acc    : {val_acc:.4f}")
        print(f"  LR         : {optimizer.param_groups[0]['lr']:.2e}")
        print(f"  Patience   : {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print(f"Early stopping après {epoch+1} epochs.")
            break

    torch.save(model.state_dict(), os.path.join(exp_dir, "final_segformer.pth"))

    history = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_ious": val_ious,
        "val_dices": val_dices,
        "val_accs": val_accs,
    }

    with open(os.path.join(exp_dir, "history.json"), "w") as f:
        json.dump(history, f)

    plot_training_curves(history, exp_dir, exp_name)

    test_results = evaluate_test_set(
        model=model,
        test_loader=test_loader,
        exp_dir=exp_dir
    )

    qualitative_visualization(
        model=model,
        test_dataset=test_dataset,
        exp_dir=exp_dir,
        num_samples=4
    )

    result = {
        "model": "SegFormer",
        "dataset": dataset_name,
        "train_size": train_size,
        "augmentation": augmentation_config,
        "best_val_iou": best_iou,
        **test_results
    }

    return result

**EVALUATE TEST SET**

In [13]:
def evaluate_test_set(model, test_loader, exp_dir=None):
    model.eval()

    test_iou = 0.0
    test_dice = 0.0
    test_acc = 0.0
    test_precision = 0.0
    test_recall = 0.0

    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Test evaluation"):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(pixel_values=images)
            logits = outputs.logits

            preds = preds_from_logits(logits, masks.shape[-2:])

            test_iou += iou_score_segformer(preds, masks).item()
            test_dice += dice_score_segformer(preds, masks).item()
            test_acc += pixel_accuracy_segformer(preds, masks).item()
            test_precision += precision_score_segformer(preds, masks).item()
            test_recall += recall_score_segformer(preds, masks).item()

    n = len(test_loader)

    results = {
        "test_iou": test_iou / n,
        "test_dice": test_dice / n,
        "test_acc": test_acc / n,
        "test_precision": test_precision / n,
        "test_recall": test_recall / n,
    }

    print("\n===== TEST RESULTS — SEGFORMER =====")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")

    if exp_dir is not None:
        with open(os.path.join(exp_dir, "test_results.json"), "w") as f:
            json.dump(results, f)

    return results

**COURBES D’ENTRAÎNEMENT**

In [14]:
def plot_training_curves(history, exp_dir, exp_name):
    epochs_range = range(1, len(history["train_losses"]) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(exp_name, fontsize=14)

    axes[0].plot(epochs_range, history["train_losses"], label="Train loss")
    axes[0].plot(epochs_range, history["val_losses"], label="Val loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs_range, history["val_ious"], label="IoU")
    axes[1].plot(epochs_range, history["val_dices"], label="Dice")
    axes[1].set_title("IoU & Dice")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(epochs_range, history["val_accs"], label="Pixel Accuracy")
    axes[2].set_title("Pixel Accuracy")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    plt.tight_layout()

    path = os.path.join(exp_dir, "training_curves.png")
    plt.savefig(path, dpi=150)
    plt.show()

    print(f"Courbes sauvegardées : {path}")

**VISUALISATION QUALITATIVE**

In [15]:
def qualitative_visualization(model, test_dataset, exp_dir, num_samples=4):
    model.eval()

    num_samples = min(num_samples, len(test_dataset))

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 4))

    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    col_titles = ["Image", "Ground Truth", "SegFormer"]

    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for i in range(num_samples):
        image, mask = test_dataset[i]

        image_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(pixel_values=image_tensor)
            logits = outputs.logits
            pred = preds_from_logits(logits, mask.shape[-2:])
            pred = pred.squeeze(0).cpu()

        image_display = denormalize(image).permute(1, 2, 0).cpu()

        iou_val = iou_score_segformer(pred, mask).item()

        axes[i, 0].imshow(image_display)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask.cpu(), cmap="gray")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(pred.cpu(), cmap="gray")
        axes[i, 2].set_xlabel(f"IoU={iou_val:.3f}", fontsize=9)
        axes[i, 2].axis("off")

    plt.tight_layout()

    path = os.path.join(exp_dir, "qualitative_segformer.png")
    plt.savefig(path, dpi=150)
    plt.show()

    print(f"Visualisation sauvegardée : {path}")

**RUN SIZE EXPERIMENTS**

In [16]:
def run_size_experiments():
    datasets = ["isic", "pets", "crack"]
    train_sizes = [50, 100, 250, 500]
    augmentation = "baseline"

    all_results = []

    for dataset_name in datasets:
        for train_size in train_sizes:

            exp_name = f"segformer_{dataset_name}_N{train_size}_{augmentation}"
            exp_dir = os.path.join(OUTPUT_DIR, exp_name)

            # CHECK SI DÉJÀ TERMINÉ
            if os.path.exists(os.path.join(exp_dir, "test_results.json")):
                print(f"✅ SKIP {exp_name} (déjà terminé)")

                # optionnel : recharger résultat
                with open(os.path.join(exp_dir, "test_results.json"), "r") as f:
                    test_results = json.load(f)

                result = {
                    "model": "SegFormer",
                    "dataset": dataset_name,
                    "train_size": train_size,
                    "augmentation": augmentation,
                    **test_results
                }

                all_results.append(result)
                continue

            print("\n" + "=" * 80)
            print(f" RUN {exp_name}")
            print("=" * 80)

            result = train_one_experiment(
                dataset_name=dataset_name,
                train_size=train_size,
                augmentation_config=augmentation
            )

            all_results.append(result)

            #  sauvegarde progressive
            df = pd.DataFrame(all_results)
            df.to_csv(os.path.join(OUTPUT_DIR, "results_size_segformer.csv"), index=False)

    return pd.DataFrame(all_results)

**RUN AUGMENTATION EXPERIMENTS**

In [17]:
def run_augmentation_experiments():
    datasets = ["isic", "pets", "crack"]
    augmentations = [
        "baseline",
        "flip",
        "flip_rotation",
        "full_aug",
        "geo_strong",
        "extreme_aug"
    ]

    train_size = 100

    all_results = []

    for dataset_name in datasets:
        for augmentation in augmentations:
            print("\n" + "=" * 80)
            print(f"RUN AUG EXPERIMENT | Dataset={dataset_name} | Aug={augmentation}")
            print("=" * 80)

            result = train_one_experiment(
                dataset_name=dataset_name,
                train_size=train_size,
                augmentation_config=augmentation
            )

            all_results.append(result)

            df = pd.DataFrame(all_results)
            df.to_csv(os.path.join(OUTPUT_DIR, "results_aug_segformer.csv"), index=False)

    return pd.DataFrame(all_results)

**LANCEMENT TEST SIMPLE**

**LANCEMENT FINAL — EXPÉRIENCES TAILLE DATASET**

**LANCEMENT FINAL — EXPÉRIENCES AUGMENTATION**

**EXPORT FINAL**

In [ ]:
print("Résultats sauvegardés dans :", OUTPUT_DIR)

!ls -R /content/segformer_results